In [1]:
"""
Supply Chain Resilience - Figure Generation Script
===================================================
Generates publication-quality figures for seminar paper and presentation.

Input: CSV files from simulation output
Output: PNG and PDF figures in specified directory
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# ============================================
# CONFIGURATION
# ============================================
INPUT_DIR = "./output_final"  # Change to your data folder
OUTPUT_DIR = "./figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Style settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 9,
    'figure.titlesize': 14,
    'font.family': 'sans-serif',
})

# Color palette for strategies
STRATEGY_COLORS = {
    'baseline': '#E74C3C',       # Red
    'dual_only': '#E67E22',      # Orange
    'safety_only': '#27AE60',    # Green
    'flex_only': '#2ECC71',      # Light Green
    'dynalloc_only': '#F1C40F',  # Yellow
    'all_combined': '#3498DB',   # Blue
}

STRATEGY_LABELS = {
    'baseline': 'Baseline',
    'dual_only': 'Dual Sourcing',
    'safety_only': 'Safety Stock',
    'flex_only': 'Flexible Capacity',
    'dynalloc_only': 'Dynamic Allocation',
    'all_combined': 'All Combined',
}

SCENARIO_LABELS = {
    'capacity_loss': 'Capacity Loss',
    'lead_time_surge': 'Lead Time Surge',
    'demand_spike': 'Demand Spike',
}

# ============================================
# LOAD DATA
# ============================================
print("Loading data...")
summary_raw = pd.read_csv(f"{INPUT_DIR}/summary_raw.csv")
timeseries = pd.read_csv(f"{INPUT_DIR}/timeseries_all.csv")
agent_data = pd.read_csv(f"{INPUT_DIR}/agent_data.csv")

# Check if we have multiple seeds
n_seeds = summary_raw['seed'].nunique()
print(f"Found {n_seeds} seed(s): {summary_raw['seed'].unique()}")

scenarios = summary_raw['scenario'].unique()
strategies = list(STRATEGY_COLORS.keys())

print(f"Scenarios: {list(scenarios)}")
print(f"Strategies: {strategies}")


# ============================================
# FIGURE 1: TIME-SERIES (FILL RATE & COST)
# ============================================
def create_timeseries_plots():
    """Create improved time-series plots for fill rate and cost"""
    
    fig, axes = plt.subplots(3, 2, figsize=(14, 12))
    
    disruption_step = 10
    
    for i, scenario in enumerate(scenarios):
        # Fill Rate subplot
        ax_fill = axes[i, 0]
        # Cost subplot
        ax_cost = axes[i, 1]
        
        for strategy in strategies:
            data = timeseries[(timeseries['scenario'] == scenario) & 
                             (timeseries['strategy'] == strategy)]
            
            if len(data) == 0:
                continue
            
            # Aggregate across seeds if multiple
            if n_seeds > 1:
                agg = data.groupby('step').agg({
                    'fill_rate': ['mean', 'std'],
                    'total_cost': ['mean', 'std']
                }).reset_index()
                agg.columns = ['step', 'fill_mean', 'fill_std', 'cost_mean', 'cost_std']
                
                ax_fill.plot(agg['step'], agg['fill_mean'], 
                           color=STRATEGY_COLORS[strategy], 
                           linewidth=2, label=STRATEGY_LABELS[strategy])
                ax_fill.fill_between(agg['step'], 
                                    agg['fill_mean'] - agg['fill_std'],
                                    agg['fill_mean'] + agg['fill_std'],
                                    color=STRATEGY_COLORS[strategy], alpha=0.15)
                
                ax_cost.plot(agg['step'], agg['cost_mean']/1000, 
                           color=STRATEGY_COLORS[strategy], 
                           linewidth=2, label=STRATEGY_LABELS[strategy])
                ax_cost.fill_between(agg['step'], 
                                    (agg['cost_mean'] - agg['cost_std'])/1000,
                                    (agg['cost_mean'] + agg['cost_std'])/1000,
                                    color=STRATEGY_COLORS[strategy], alpha=0.15)
            else:
                ax_fill.plot(data['step'], data['fill_rate'], 
                           color=STRATEGY_COLORS[strategy], 
                           linewidth=2, label=STRATEGY_LABELS[strategy])
                ax_cost.plot(data['step'], data['total_cost']/1000, 
                           color=STRATEGY_COLORS[strategy], 
                           linewidth=2, label=STRATEGY_LABELS[strategy])
        
        # Disruption line
        ax_fill.axvline(x=disruption_step, color='red', linestyle='--', 
                       linewidth=2, alpha=0.7)
        ax_cost.axvline(x=disruption_step, color='red', linestyle='--', 
                       linewidth=2, alpha=0.7)
        
        # Formatting
        ax_fill.set_title(f'Fill Rate — {SCENARIO_LABELS[scenario]}', fontweight='bold')
        ax_fill.set_xlabel('Time Step')
        ax_fill.set_ylabel('Fill Rate')
        ax_fill.set_ylim(0.65, 1.05)
        ax_fill.set_xlim(0, timeseries['step'].max())
        
        ax_cost.set_title(f'Cumulative Cost — {SCENARIO_LABELS[scenario]}', fontweight='bold')
        ax_cost.set_xlabel('Time Step')
        ax_cost.set_ylabel('Cost (×1,000)')
        ax_cost.set_xlim(0, timeseries['step'].max())
        
        # Add legend only to first row
        if i == 0:
            ax_fill.legend(loc='lower left', fontsize=8, framealpha=0.9)
            ax_cost.legend(loc='upper left', fontsize=8, framealpha=0.9)
        
        # Add disruption annotation
        if i == 0:
            ax_fill.annotate('Disruption', xy=(disruption_step, 1.02), 
                           fontsize=9, color='red', ha='center')
    
    plt.tight_layout()
    return fig


# ============================================
# FIGURE 2: TTR BAR CHART (BY SCENARIO)
# ============================================
def create_ttr_comparison():
    """Create TTR comparison bar chart grouped by scenario"""
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Aggregate data
    agg = summary_raw.groupby(['scenario', 'strategy']).agg({
        'ttr': ['mean', 'std']
    }).reset_index()
    agg.columns = ['scenario', 'strategy', 'ttr_mean', 'ttr_std']
    
    x = np.arange(len(scenarios))
    width = 0.13
    multiplier = 0
    
    for strategy in strategies:
        subset = agg[agg['strategy'] == strategy]
        # Ensure correct order
        subset = subset.set_index('scenario').reindex(scenarios).reset_index()
        
        offset = width * multiplier
        bars = ax.bar(x + offset, subset['ttr_mean'], width, 
                     label=STRATEGY_LABELS[strategy],
                     color=STRATEGY_COLORS[strategy],
                     edgecolor='white', linewidth=0.5)
        
        # Error bars if multiple seeds
        if n_seeds > 1:
            ax.errorbar(x + offset, subset['ttr_mean'], yerr=subset['ttr_std'],
                       fmt='none', color='black', capsize=3, capthick=1, linewidth=1)
        
        multiplier += 1
    
    ax.set_xlabel('Disruption Scenario', fontweight='bold')
    ax.set_ylabel('Time to Recovery (periods)', fontweight='bold')
    ax.set_title('Time to Recovery by Strategy and Scenario', fontsize=14, fontweight='bold')
    ax.set_xticks(x + width * 2.5)
    ax.set_xticklabels([SCENARIO_LABELS[s] for s in scenarios])
    ax.legend(loc='upper right', ncol=2, framealpha=0.9)
    ax.set_ylim(0, ax.get_ylim()[1] * 1.15)
    
    # Add gridlines
    ax.yaxis.grid(True, linestyle='--', alpha=0.7)
    ax.set_axisbelow(True)
    
    plt.tight_layout()
    return fig


# ============================================
# FIGURE 3: TTR HORIZONTAL BAR (AVERAGED)
# ============================================
def create_ttr_horizontal():
    """Create horizontal bar chart for TTR averaged across scenarios"""
    
    fig, ax = plt.subplots(figsize=(10, 5))
    
    # Aggregate across all scenarios
    agg = summary_raw.groupby('strategy').agg({
        'ttr': ['mean', 'std']
    }).reset_index()
    agg.columns = ['strategy', 'ttr_mean', 'ttr_std']
    
    # Sort by TTR descending
    agg = agg.sort_values('ttr_mean', ascending=True)
    
    y_pos = np.arange(len(agg))
    colors = [STRATEGY_COLORS[s] for s in agg['strategy']]
    
    bars = ax.barh(y_pos, agg['ttr_mean'], color=colors, 
                   edgecolor='white', height=0.7)
    
    # Error bars
    if n_seeds > 1:
        ax.errorbar(agg['ttr_mean'], y_pos, xerr=agg['ttr_std'],
                   fmt='none', color='black', capsize=4, capthick=1)
    
    # Value labels
    for i, (bar, val, std) in enumerate(zip(bars, agg['ttr_mean'], agg['ttr_std'])):
        if n_seeds > 1:
            label = f'{val:.1f} ± {std:.1f}'
        else:
            label = f'{val:.1f}'
        
        if val > 20:
            ax.text(val - 2, bar.get_y() + bar.get_height()/2, 
                   label, ha='right', va='center', 
                   fontsize=10, fontweight='bold', color='white')
        else:
            ax.text(val + 2, bar.get_y() + bar.get_height()/2, 
                   label, ha='left', va='center', 
                   fontsize=10, fontweight='bold', color='#333333')
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels([STRATEGY_LABELS[s] for s in agg['strategy']])
    ax.set_xlabel('Average Time to Recovery (periods)', fontweight='bold')
    ax.set_title('Strategy Comparison: Time to Recovery\n(Averaged Across All Scenarios)', 
                fontsize=13, fontweight='bold')
    
    ax.xaxis.grid(True, linestyle='--', alpha=0.7)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    return fig


# ============================================
# FIGURE 4: COST-SERVICE SCATTER PLOT
# ============================================
def create_cost_service_scatter():
    """Create cost vs fill rate scatter plot with TTR as bubble size"""
    
    fig, ax = plt.subplots(figsize=(11, 7))
    
    # Aggregate data
    agg = summary_raw.groupby('strategy').agg({
        'post_fill': 'mean',
        'final_cost': 'mean',
        'ttr': 'mean'
    }).reset_index()
    
    # Bubble size: inverse TTR (faster recovery = larger bubble)
    max_ttr = agg['ttr'].max()
    bubble_sizes = [(max_ttr - t + 10) * 12 for t in agg['ttr']]
    
    # Scatter plot
    for i, row in agg.iterrows():
        strategy = row['strategy']
        ax.scatter(row['post_fill'], row['final_cost']/1000, 
                  s=bubble_sizes[i], 
                  c=STRATEGY_COLORS[strategy], 
                  alpha=0.7,
                  edgecolors='white',
                  linewidths=2,
                  label=STRATEGY_LABELS[strategy],
                  zorder=3)
    
    # Add labels
    label_offsets = {
        'baseline': (0.005, 3),
        'dual_only': (0.005, -5),
        'safety_only': (-0.02, -4),
        'flex_only': (-0.02, 3),
        'dynalloc_only': (0.005, 0),
        'all_combined': (0.005, 3),
    }
    
    for i, row in agg.iterrows():
        strategy = row['strategy']
        offset = label_offsets.get(strategy, (0.005, 0))
        ax.annotate(STRATEGY_LABELS[strategy], 
                   xy=(row['post_fill'], row['final_cost']/1000),
                   xytext=(row['post_fill'] + offset[0], row['final_cost']/1000 + offset[1]),
                   fontsize=9, fontweight='bold',
                   ha='left' if offset[0] > 0 else 'right')
    
    ax.set_xlabel('Average Fill Rate (Post-Disruption)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Average Total Cost (×1,000)', fontsize=12, fontweight='bold')
    ax.set_title('Cost-Service Trade-off Analysis\n(Bubble size ∝ recovery speed)', 
                fontsize=13, fontweight='bold')
    
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Direction arrows
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    ax.annotate('', xy=(xlim[1], ylim[0] + 2), xytext=(xlim[0] + 0.02, ylim[0] + 2),
               arrowprops=dict(arrowstyle='->', color='#666666', lw=1.5))
    ax.text((xlim[0] + xlim[1])/2, ylim[0] - 1, 'Higher Fill Rate →', 
           fontsize=9, ha='center', color='#666666')
    
    plt.tight_layout()
    return fig


# ============================================
# FIGURE 5: HEATMAP (STRATEGY × SCENARIO)
# ============================================
def create_performance_heatmap():
    """Create heatmap showing performance across strategies and scenarios"""
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    metrics = [
        ('post_fill', 'Fill Rate', 'RdYlGn', '{:.3f}'),
        ('ttr', 'Time to Recovery', 'RdYlGn_r', '{:.0f}'),
        ('final_cost', 'Total Cost (×1k)', 'RdYlGn_r', '{:.0f}'),
    ]
    
    for ax, (metric, title, cmap, fmt) in zip(axes, metrics):
        # Pivot table
        pivot = summary_raw.pivot_table(
            values=metric, 
            index='strategy', 
            columns='scenario',
            aggfunc='mean'
        )
        
        # Reorder
        pivot = pivot.reindex(strategies)
        pivot = pivot[[s for s in scenarios if s in pivot.columns]]
        
        # Rename for display
        pivot.index = [STRATEGY_LABELS[s] for s in pivot.index]
        pivot.columns = [SCENARIO_LABELS[s] for s in pivot.columns]
        
        # Scale cost for display
        if metric == 'final_cost':
            pivot = pivot / 1000
        
        # Plot heatmap
        sns.heatmap(pivot, annot=True, fmt=fmt.replace('{:', '').replace('}', ''),
                   cmap=cmap, ax=ax, cbar=True,
                   linewidths=0.5, linecolor='white')
        
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlabel('')
        ax.set_ylabel('')
    
    plt.suptitle('Performance Comparison: Strategy × Scenario', 
                fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    return fig


# ============================================
# FIGURE 6: FILL RATE DROP COMPARISON
# ============================================
def create_fill_drop_comparison():
    """Create bar chart showing fill rate drop by strategy and scenario"""
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    agg = summary_raw.groupby(['scenario', 'strategy']).agg({
        'fill_drop_pct': ['mean', 'std']
    }).reset_index()
    agg.columns = ['scenario', 'strategy', 'drop_mean', 'drop_std']
    
    x = np.arange(len(scenarios))
    width = 0.13
    multiplier = 0
    
    for strategy in strategies:
        subset = agg[agg['strategy'] == strategy]
        subset = subset.set_index('scenario').reindex(scenarios).reset_index()
        
        offset = width * multiplier
        bars = ax.bar(x + offset, subset['drop_mean'], width, 
                     label=STRATEGY_LABELS[strategy],
                     color=STRATEGY_COLORS[strategy],
                     edgecolor='white', linewidth=0.5)
        
        if n_seeds > 1:
            ax.errorbar(x + offset, subset['drop_mean'], yerr=subset['drop_std'],
                       fmt='none', color='black', capsize=3, capthick=1)
        
        multiplier += 1
    
    ax.set_xlabel('Disruption Scenario', fontweight='bold')
    ax.set_ylabel('Fill Rate Drop (%)', fontweight='bold')
    ax.set_title('Service Level Impact: Fill Rate Drop During Disruption', 
                fontsize=13, fontweight='bold')
    ax.set_xticks(x + width * 2.5)
    ax.set_xticklabels([SCENARIO_LABELS[s] for s in scenarios])
    ax.legend(loc='upper right', ncol=2, framealpha=0.9)
    
    ax.yaxis.grid(True, linestyle='--', alpha=0.7)
    ax.set_axisbelow(True)
    
    plt.tight_layout()
    return fig


# ============================================
# FIGURE 7: COST BREAKDOWN (HOLDING VS BACKLOG)
# ============================================
def create_cost_breakdown():
    """Create stacked bar chart showing holding vs backlog cost"""
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    agg = summary_raw.groupby('strategy').agg({
        'holding_cost': 'mean',
        'backlog_cost': 'mean'
    }).reset_index()
    
    # Reorder
    agg = agg.set_index('strategy').reindex(strategies).reset_index()
    
    x = np.arange(len(strategies))
    width = 0.6
    
    bars1 = ax.bar(x, agg['holding_cost']/1000, width, 
                   label='Holding Cost', color='#3498DB', edgecolor='white')
    bars2 = ax.bar(x, agg['backlog_cost']/1000, width, 
                   bottom=agg['holding_cost']/1000,
                   label='Backlog Cost', color='#E74C3C', edgecolor='white')
    
    # Add total labels on top
    for i, (h, b) in enumerate(zip(agg['holding_cost'], agg['backlog_cost'])):
        total = (h + b) / 1000
        ax.text(i, total + 2, f'{total:.0f}k', ha='center', va='bottom', 
               fontsize=9, fontweight='bold')
    
    ax.set_xlabel('Strategy', fontweight='bold')
    ax.set_ylabel('Cost (×1,000)', fontweight='bold')
    ax.set_title('Cost Breakdown by Strategy\n(Averaged Across Scenarios)', 
                fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([STRATEGY_LABELS[s] for s in strategies], rotation=15, ha='right')
    ax.legend(loc='upper left', framealpha=0.9)
    
    ax.yaxis.grid(True, linestyle='--', alpha=0.7)
    ax.set_axisbelow(True)
    
    plt.tight_layout()
    return fig


# ============================================
# FIGURE 8: BULLWHIP BY TIER
# ============================================
def create_bullwhip_by_tier():
    """Create grouped bar chart showing bullwhip ratio by tier"""
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    tiers = ['retailer', 'dc', 'plant', 'supplier']
    tier_labels = ['Retailer', 'DC', 'Plant', 'Supplier']
    
    agg = summary_raw.groupby('strategy').agg({
        'bullwhip_retailer': 'mean',
        'bullwhip_dc': 'mean',
        'bullwhip_plant': 'mean',
        'bullwhip_supplier': 'mean',
    }).reset_index()
    
    agg = agg.set_index('strategy').reindex(strategies).reset_index()
    
    x = np.arange(len(strategies))
    width = 0.18
    
    tier_colors = ['#FF6B6B', '#F9A826', '#45B7D1', '#4ECDC4']
    
    for i, (tier, color, label) in enumerate(zip(tiers, tier_colors, tier_labels)):
        col = f'bullwhip_{tier}'
        offset = width * i
        ax.bar(x + offset, agg[col], width, label=label, color=color, edgecolor='white')
    
    ax.set_xlabel('Strategy', fontweight='bold')
    ax.set_ylabel('Bullwhip Ratio', fontweight='bold')
    ax.set_title('Bullwhip Effect by Supply Chain Tier', fontsize=13, fontweight='bold')
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels([STRATEGY_LABELS[s] for s in strategies], rotation=15, ha='right')
    ax.legend(loc='upper right', framealpha=0.9)
    
    ax.yaxis.grid(True, linestyle='--', alpha=0.7)
    ax.set_axisbelow(True)
    ax.set_yscale('log')  # Log scale due to large variation
    
    plt.tight_layout()
    return fig


# ============================================
# TABLE: LATEX SUMMARY TABLE
# ============================================
def generate_latex_tables():
    """Generate LaTeX tables for paper"""
    
    tables = {}
    
    # Table 1: Main results by scenario
    for scenario in scenarios:
        subset = summary_raw[summary_raw['scenario'] == scenario]
        agg = subset.groupby('strategy').agg({
            'post_fill': ['mean', 'std'],
            'min_fill': ['mean', 'std'],
            'final_cost': ['mean', 'std'],
            'ttr': ['mean', 'std'],
        })
        
        latex = "\\begin{tabular}{lcccc}\n"
        latex += "\\toprule\n"
        latex += "Strategy & Fill Rate & Min Fill & Cost (×1k) & TTR \\\\\n"
        latex += "\\midrule\n"
        
        for strategy in strategies:
            if strategy in agg.index:
                row = agg.loc[strategy]
                if n_seeds > 1:
                    latex += f"{STRATEGY_LABELS[strategy]} & "
                    latex += f"{row[('post_fill', 'mean')]:.3f} ± {row[('post_fill', 'std')]:.3f} & "
                    latex += f"{row[('min_fill', 'mean')]:.3f} ± {row[('min_fill', 'std')]:.3f} & "
                    latex += f"{row[('final_cost', 'mean')]/1000:.0f} ± {row[('final_cost', 'std')]/1000:.0f} & "
                    latex += f"{row[('ttr', 'mean')]:.0f} ± {row[('ttr', 'std')]:.0f} \\\\\n"
                else:
                    latex += f"{STRATEGY_LABELS[strategy]} & "
                    latex += f"{row[('post_fill', 'mean')]:.3f} & "
                    latex += f"{row[('min_fill', 'mean')]:.3f} & "
                    latex += f"{row[('final_cost', 'mean')]/1000:.0f} & "
                    latex += f"{row[('ttr', 'mean')]:.0f} \\\\\n"
        
        latex += "\\bottomrule\n"
        latex += "\\end{tabular}"
        
        tables[scenario] = latex
    
    # Table 2: Cross-scenario summary
    agg_all = summary_raw.groupby('strategy').agg({
        'post_fill': ['mean', 'std'],
        'final_cost': ['mean', 'std'],
        'ttr': ['mean', 'std'],
        'bullwhip': ['mean', 'std'],
    })
    
    latex_summary = "\\begin{tabular}{lcccc}\n"
    latex_summary += "\\toprule\n"
    latex_summary += "Strategy & Fill Rate & Cost (×1k) & TTR & Bullwhip \\\\\n"
    latex_summary += "\\midrule\n"
    
    for strategy in strategies:
        if strategy in agg_all.index:
            row = agg_all.loc[strategy]
            if n_seeds > 1:
                latex_summary += f"{STRATEGY_LABELS[strategy]} & "
                latex_summary += f"{row[('post_fill', 'mean')]:.3f} ± {row[('post_fill', 'std')]:.3f} & "
                latex_summary += f"{row[('final_cost', 'mean')]/1000:.0f} ± {row[('final_cost', 'std')]/1000:.0f} & "
                latex_summary += f"{row[('ttr', 'mean')]:.0f} ± {row[('ttr', 'std')]:.0f} & "
                latex_summary += f"{row[('bullwhip', 'mean')]:.3f} ± {row[('bullwhip', 'std')]:.3f} \\\\\n"
            else:
                latex_summary += f"{STRATEGY_LABELS[strategy]} & "
                latex_summary += f"{row[('post_fill', 'mean')]:.3f} & "
                latex_summary += f"{row[('final_cost', 'mean')]/1000:.0f} & "
                latex_summary += f"{row[('ttr', 'mean')]:.0f} & "
                latex_summary += f"{row[('bullwhip', 'mean')]:.3f} \\\\\n"
    
    latex_summary += "\\bottomrule\n"
    latex_summary += "\\end{tabular}"
    
    tables['summary'] = latex_summary
    
    return tables


# ============================================
# MAIN: GENERATE ALL FIGURES
# ============================================
if __name__ == "__main__":
    
    print("\n" + "="*60)
    print("GENERATING FIGURES")
    print("="*60)
    print(f"Output directory: {OUTPUT_DIR}\n")
    
    # Figure 1: Time-series
    print("Creating Figure 1: Time-Series Plots...")
    fig1 = create_timeseries_plots()
    fig1.savefig(f"{OUTPUT_DIR}/fig1_timeseries.png", dpi=300, bbox_inches='tight', 
                facecolor='white', edgecolor='none')
    fig1.savefig(f"{OUTPUT_DIR}/fig1_timeseries.pdf", bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.close(fig1)
    print("  ✓ Saved fig1_timeseries.png/pdf")
    
    # Figure 2: TTR by scenario
    print("Creating Figure 2: TTR Comparison by Scenario...")
    fig2 = create_ttr_comparison()
    fig2.savefig(f"{OUTPUT_DIR}/fig2_ttr_by_scenario.png", dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    fig2.savefig(f"{OUTPUT_DIR}/fig2_ttr_by_scenario.pdf", bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.close(fig2)
    print("  ✓ Saved fig2_ttr_by_scenario.png/pdf")
    
    # Figure 3: TTR horizontal (averaged)
    print("Creating Figure 3: TTR Horizontal Bar...")
    fig3 = create_ttr_horizontal()
    fig3.savefig(f"{OUTPUT_DIR}/fig3_ttr_horizontal.png", dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    fig3.savefig(f"{OUTPUT_DIR}/fig3_ttr_horizontal.pdf", bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.close(fig3)
    print("  ✓ Saved fig3_ttr_horizontal.png/pdf")
    
    # Figure 4: Cost-Service scatter
    print("Creating Figure 4: Cost-Service Scatter...")
    fig4 = create_cost_service_scatter()
    fig4.savefig(f"{OUTPUT_DIR}/fig4_cost_service.png", dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    fig4.savefig(f"{OUTPUT_DIR}/fig4_cost_service.pdf", bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.close(fig4)
    print("  ✓ Saved fig4_cost_service.png/pdf")
    
    # Figure 5: Heatmap
    print("Creating Figure 5: Performance Heatmap...")
    fig5 = create_performance_heatmap()
    fig5.savefig(f"{OUTPUT_DIR}/fig5_heatmap.png", dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    fig5.savefig(f"{OUTPUT_DIR}/fig5_heatmap.pdf", bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.close(fig5)
    print("  ✓ Saved fig5_heatmap.png/pdf")
    
    # Figure 6: Fill Rate Drop
    print("Creating Figure 6: Fill Rate Drop...")
    fig6 = create_fill_drop_comparison()
    fig6.savefig(f"{OUTPUT_DIR}/fig6_fill_drop.png", dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    fig6.savefig(f"{OUTPUT_DIR}/fig6_fill_drop.pdf", bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.close(fig6)
    print("  ✓ Saved fig6_fill_drop.png/pdf")
    
    # Figure 7: Cost Breakdown
    print("Creating Figure 7: Cost Breakdown...")
    fig7 = create_cost_breakdown()
    fig7.savefig(f"{OUTPUT_DIR}/fig7_cost_breakdown.png", dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    fig7.savefig(f"{OUTPUT_DIR}/fig7_cost_breakdown.pdf", bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.close(fig7)
    print("  ✓ Saved fig7_cost_breakdown.png/pdf")
    
    # Figure 8: Bullwhip by Tier
    print("Creating Figure 8: Bullwhip by Tier...")
    fig8 = create_bullwhip_by_tier()
    fig8.savefig(f"{OUTPUT_DIR}/fig8_bullwhip_tier.png", dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    fig8.savefig(f"{OUTPUT_DIR}/fig8_bullwhip_tier.pdf", bbox_inches='tight',
                facecolor='white', edgecolor='none')
    plt.close(fig8)
    print("  ✓ Saved fig8_bullwhip_tier.png/pdf")
    
    # LaTeX Tables
    print("\nGenerating LaTeX Tables...")
    tables = generate_latex_tables()
    
    with open(f"{OUTPUT_DIR}/tables.tex", 'w') as f:
        f.write("% Auto-generated LaTeX tables\n")
        f.write("% Supply Chain Resilience Simulation Results\n\n")
        
        for name, latex in tables.items():
            f.write(f"% Table: {name}\n")
            f.write(latex)
            f.write("\n\n")
    
    print("  ✓ Saved tables.tex")
    
    # Summary
    print("\n" + "="*60)
    print("COMPLETE")
    print("="*60)
    print(f"\nAll files saved to: {OUTPUT_DIR}/")
    print("\nFigures:")
    print("  - fig1_timeseries.png/pdf      (Fill rate & cost over time)")
    print("  - fig2_ttr_by_scenario.png/pdf (TTR grouped by scenario)")
    print("  - fig3_ttr_horizontal.png/pdf  (TTR averaged, horizontal bars)")
    print("  - fig4_cost_service.png/pdf    (Cost vs Fill Rate scatter)")
    print("  - fig5_heatmap.png/pdf         (Strategy × Scenario heatmap)")
    print("  - fig6_fill_drop.png/pdf       (Fill rate drop comparison)")
    print("  - fig7_cost_breakdown.png/pdf  (Holding vs Backlog cost)")
    print("  - fig8_bullwhip_tier.png/pdf   (Bullwhip by tier)")
    print("\nTables:")
    print("  - tables.tex                   (LaTeX tables for paper)")

Loading data...
Found 3 seed(s): [2411   24  200]
Scenarios: ['capacity_loss', 'lead_time_surge', 'demand_spike']
Strategies: ['baseline', 'dual_only', 'safety_only', 'flex_only', 'dynalloc_only', 'all_combined']

GENERATING FIGURES
Output directory: ./figures

Creating Figure 1: Time-Series Plots...
  ✓ Saved fig1_timeseries.png/pdf
Creating Figure 2: TTR Comparison by Scenario...
  ✓ Saved fig2_ttr_by_scenario.png/pdf
Creating Figure 3: TTR Horizontal Bar...
  ✓ Saved fig3_ttr_horizontal.png/pdf
Creating Figure 4: Cost-Service Scatter...
  ✓ Saved fig4_cost_service.png/pdf
Creating Figure 5: Performance Heatmap...
  ✓ Saved fig5_heatmap.png/pdf
Creating Figure 6: Fill Rate Drop...
  ✓ Saved fig6_fill_drop.png/pdf
Creating Figure 7: Cost Breakdown...
  ✓ Saved fig7_cost_breakdown.png/pdf
Creating Figure 8: Bullwhip by Tier...
  ✓ Saved fig8_bullwhip_tier.png/pdf

Generating LaTeX Tables...
  ✓ Saved tables.tex

COMPLETE

All files saved to: ./figures/

Figures:
  - fig1_timeseries.png